# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [3]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup complete!


In [4]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [5]:
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [6]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

Gold: 100, Weak: 220, LLM: 150


In [7]:
#  1c. Filter LLM labels by confidence 
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label
X_llm = vec.transform(llm["review"])
probs = clf.predict_proba(X_llm)
preds = clf.predict(X_llm)

confidence = np.max(probs, axis=1)
llm["predicted_label"] = preds
llm["confidence"] = confidence

filtered_llm = llm[(llm["confidence"] >= 0.65) & (llm["predicted_label"] == llm["label"])]
filtered_llm = filtered_llm.drop(columns=["predicted_label", "confidence"])

print(f"Filtered LLM: {len(filtered_llm)} out of {len(llm)}")

Filtered LLM: 27 out of 150


In [8]:
#  1d. Merge & deduplicate 
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
consolidated_df = pd.concat([gold, weak, filtered_llm], ignore_index=True)
# removing exact duplicates based on the "review" column
consolidated_df = consolidated_df.drop_duplicates(subset="review")
consolidated_df.to_csv("consolidated_base.csv", index=False)

print(f"Consolidated dataset: {len(consolidated_df)} reviews")

Consolidated dataset: 328 reviews


In [9]:
#  1e. Class distribution analysis 
# TODO: Count per class, identify minority, plot and save class_distribution.png
class_counts = consolidated_df["label"].value_counts()
minority_class = class_counts.idxmin()
print("Class distribution:\n", class_counts)
print("Minority class:", minority_class)

class_counts.plot(kind="bar", color=["blue", "orange", "green"])
plt.title("Class Distribution in Consolidated Dataset")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.savefig("class_distribution.png")
print("Class distribution plot saved as class_distribution.png")

Class distribution:
 label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64
Minority class: Positive
Class distribution plot saved as class_distribution.png


In [10]:
#  1f. Augmentation functions 

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    # TODO: Implement using nltk.corpus.wordnet, nltk.pos_tag, word_tokenize
    words = word_tokenize(text)
    #Getting Part of Speech tags for the words in the text, ie, if the word is a noun, verb, adjective or adverb
    pos_tags = pos_tag(words)
    new_words = words.copy()

    #List of words that we can potentially change without risking any sentiment change or grammatical issues or meaning loss
    potentially_replaceable = [
        i for i, (word, pos) in enumerate(pos_tags)
        if word.lower() not in PRESERVE_WORDS and word.isalpha() and pos.startswith(("NN", "VB", "JJ", "RB"))
    ]   #word.isalpha() ensures we only consider actual words, not punctuation or numbers

    num_to_replace = max(1, int(len(words) * replace_frac))
    to_replace = random.sample(potentially_replaceable, min(num_to_replace, len(potentially_replaceable)))
    for idx in to_replace:
        word = words[idx]
        synonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                synonym = lemma.name().replace("_", " ")
                if synonym.lower() != word.lower():
                    synonyms.add(synonym)
        if synonyms:   
            new_words[idx] = random.choice(list(synonyms))
    return " ".join(new_words)

def back_translate(text, src="en", mid="hi"):
    """Translate English $\rightarrow$ Hindi $\rightarrow$ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    # TODO: Implement with error handling and rate-limit sleep
    try:
        time.sleep(0.5)  # Sleep to respect rate limits
        to_mid = GoogleTranslator(source=src, target=mid)
        from_mid = GoogleTranslator(source=mid, target=src)
        mid_text = to_mid.translate(text)
        back_text = from_mid.translate(mid_text)
        return back_text
    except Exception as e:
        print(f"Back-translation error: {e}")
        return text  # Return original if translation fails

def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    # TODO: Implement Jaccard similarity check
    original_set = set(original.lower().split())
    augmented_set = set(augmented.lower().split())
    intersection = original_set.intersection(augmented_set)
    union = original_set.union(augmented_set)
    jaccard_sim = len(intersection) / len(union) if union else 0
    return 0.30 <= jaccard_sim <= 0.95


In [11]:
#  1g. Apply augmentation to minority class 
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
minority_samples = consolidated_df[consolidated_df["label"] == minority_class]
augmented_texts = []
for _, row in tqdm(minority_samples.iterrows(), total=len(minority_samples), desc="Augmenting minority class"):
    original = row["review"]
    aug_syn = synonym_replacement(original)
    aug_bt = back_translate(original)
# TODO: Apply quality_filter, keep only passing samples
    if quality_filter(original, aug_syn):
        augmented_texts.append((original, aug_syn, row["label"]))
    if quality_filter(original, aug_bt):
        augmented_texts.append((original, aug_bt, row["label"]))
# TODO: Save augmented_classical.csv
augmented_df = pd.DataFrame(augmented_texts, columns=["original_review", "augmented_review", "label"])
augmented_df.to_csv("augmented_classical.csv", index=False)
print(f"Augmented samples saved: {len(augmented_df)}")

Augmenting minority class: 100%|██████████| 62/62 [01:53<00:00,  1.83s/it]

Augmented samples saved: 123


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [12]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt 
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
PROMPT_TEMPLATE = """You are a movie review generator for a sentiment analysis task. Generate a batch of 20 movie reviews. Some examples are given below. Tend to generate different styles of reviews, but ensure correct labels.
Out of 20, generate 10 Positive, 6-7 Negative, and remaining Neutral reviews. Output the reviews in JSON format as a list of objects with "review", "sentiment", and "movie" fields. Here are some examples:

1. review: "This movie is a triumph in every sense. Highly recommended for everyone", sentiment: "Positive"
2. review: "I have never been so bored in my life. The score was frankly cringe-worthy. Avoid this at all costs.", sentiment: "Negative"
3. review: ""Middle of the road entertainment. Visually it's fine, but the story arc is just average. It is what it is", sentiment: "Neutral"

Requirements:
- Ensure the JSON is well-formed and parsable.
- Vary the writing style across reviews (e.g., some can be formal, others more casual or humorous).
- Follow the specified sentiment distribution closely (10 Positive, 6-7 Negative, rest Neutral).
- Generate a batch of 20 reviews
- Sentiments should be one of "Positive", "Negative", or "Neutral".

Format: 

[{"review": "...", "sentiment": "Positive", "movie": "..."}, {}......]

"""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)


In [13]:
#Function to validate generated reviews from LLM
def validate_review(review):
    """Check if review has required fields and valid sentiment."""
    if not isinstance(review, dict):
        return False
    if "review" not in review or "sentiment" not in review:
        return False
    if review["sentiment"] not in {"Positive", "Negative", "Neutral"}:
        return False
    return True

In [14]:
#  2b. Generate reviews in batches 
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors
generated_reviews = []
for i in tqdm(range(15), desc="Generating reviews in batches"):
    try:
        response = client.chat.completions.create(
            model = LLM_MODEL,
            messages = [{"role": "user", "content": PROMPT_TEMPLATE}],
            response_format={"type": "json_object"}
        )

        content = response.choices[0].message.content
        batch_reviews = json.loads(content)
        
        if isinstance(batch_reviews, list):
            valid_count = 0
            for item in batch_reviews:
                if validate_review(item):
                    generated_reviews.append(item)
                    valid_count += 1
            print(f"Batch {i+1}: {valid_count} valid reviews generated.")
        time.sleep(1)  # Sleep to respect rate limits
    
    except Exception as e:
        print(f"Error during generation: {e}")
        time.sleep(5)  # Longer sleep on error

llm_gen_df = pd.DataFrame(generated_reviews)
llm_gen_df = llm_gen_df.rename(columns={'sentiment': 'label'})

Generating reviews in batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batch 1: 20 valid reviews generated.


Generating reviews in batches:   7%|▋         | 1/15 [00:08<01:57,  8.40s/it]

Batch 2: 20 valid reviews generated.


Generating reviews in batches:  13%|█▎        | 2/15 [00:16<01:46,  8.16s/it]

...

Generating reviews in batches:  87%|████████▋ | 13/15 [01:45<00:16,  8.27s/it]

Batch 14: 20 valid reviews generated.


Generating reviews in batches:  93%|█████████▎| 14/15 [01:54<00:08,  8.25s/it]

Batch 15: 20 valid reviews generated.


Generating reviews in batches: 100%|██████████| 15/15 [02:02<00:00,  8.14s/it]


In [15]:
#  2c. Diversity metrics 
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
class_scores = {}
for label in ['Positive', 'Negative', 'Neutral']:
    reviews = llm_gen_df[llm_gen_df["label"] == label]["review"].tolist()
    bleu_scores = []
    tokenized_reviews = [word_tokenize(r.lower()) for r in reviews]
    smoothie = SmoothingFunction().method1 # Prevents 0 scores for short reviews
    sample_size = min(50, len(tokenized_reviews))
    sample_indices = random.sample(range(len(tokenized_reviews)), sample_size)
    for i in sample_indices:
        hypothesis = tokenized_reviews[i]
        references = [tokenized_reviews[j] for j in range(len(tokenized_reviews)) if i != j] # The rest
        
        score = sentence_bleu(references, hypothesis, smoothing_function=smoothie)
        bleu_scores.append(score)
    avg_bleu = np.mean(bleu_scores) if bleu_scores else 0
    class_scores[label] = float(avg_bleu)
print("Self-BLEU scores by class:", class_scores)

Self-BLEU scores by class: {'Positive': 0.5330363561985179, 'Negative': 0.5126771822997628, 'Neutral': 0.46541333374767874}


In [16]:
#  2d. Sentiment consistency check 
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv
X_gen = vec.transform(llm_gen_df['review'])
gen_preds = clf.predict(X_gen)
llm_gen_df['predicted_label'] = gen_preds
mismatches = llm_gen_df[llm_gen_df['predicted_label'] != llm_gen_df['label']]
print(f"Sentiment mismatches: {len(mismatches)} out of {len(llm_gen_df)}")
mismatches.to_csv("llm_generated_flagged.csv", index=False)

Sentiment mismatches: 133 out of 300


In the mismatches found above, a majority of them if not all have the vectoriser and classfier giving the wrong instead of the LLM being incorrect. This is because our models have been trained on few hundereds of reviews, while the internal vocabulary of LLM (Gemini flash) is way to large. Thus, this leads to wrong predictions by our logistic regression model for the reviews generated by LLM.

In [17]:
#  2e. Save outputs 
# TODO: Save llm_generated_300.csv and diversity_report.txt
llm_gen_df.drop(columns=["predicted_label"], inplace=True)
llm_gen_df.to_csv("llm_generated_300.csv", index=False)

with open("diversity_report.txt", "w", encoding="utf-8") as f:
    f.write("Diversity Report for LLM-Generated Reviews\n")
    f.write("=========================================\n\n")
    f.write(f"Total reviews generated: {len(llm_gen_df)}\n")
    f.write(f"Class distribution:\n{llm_gen_df['label'].value_counts()}\n\n")
    f.write("Self-BLEU scores by class:\n")
    for label, score in class_scores.items():
        f.write(f"{label}: {score:.4f}\n")

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [18]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling 
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)
consolidated_df["length"] = consolidated_df["review"].str.len()
sampled_df = pd.DataFrame()

for label, count in [("Positive", 40), ("Negative", 40), ("Neutral", 20)]:
    subset = consolidated_df[consolidated_df["label"] == label].sort_values("length").head(count)
    sampled_df = pd.concat([sampled_df, subset], ignore_index=True)

sampled_df.drop(columns=["length"], inplace=True)

print(f"Sampled {len(sampled_df)} reviews for translation.")
print(sampled_df['label'].value_counts())

Sampled 100 reviews for translation.
label
Positive    40
Negative    40
Neutral     20
Name: count, dtype: int64


In [19]:
en_to_hi = GoogleTranslator(source='en', target='hi')
hi_to_en = GoogleTranslator(source='hi', target='en')

results = []
smoothie = SmoothingFunction().method1

for idx, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Translating reviews"):
    original = row["review"]
    label = row["label"]
    try:
        time.sleep(0.5)  # Sleep to respect rate limits

#  3b. Translation pipeline 
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits

        hindi = en_to_hi.translate(original)
        time.sleep(0.5)  # Sleep before back-translation

#  3c. Back-translation & BLEU 
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"

        back_translated = hi_to_en.translate(hindi)
        ref = [word_tokenize(original.lower())]
        hyp = word_tokenize(back_translated.lower())
        bleu_score = sentence_bleu(ref, hyp, smoothing_function=smoothie)
        quality_flag = "PASS" if bleu_score >= 0.3 else "FAIL"

#  3d. Sentiment preservation check 
# TODO: Predict sentiment on back-translated text, compare with original label

        sentiment_pred = clf.predict(vec.transform([back_translated]))[0]
        if sentiment_pred != label:
            quality_flag = "FAIL"
        results.append({
            "review": original,
            "label": label,
            "hindi": hindi,
            "back_translated": back_translated,
            "bleu_score": bleu_score,
            "quality_flag": quality_flag
        })

    except Exception as e:
        print(f"Translation Error at index {idx}: {e}")
        results.append({
            "review": original, "label": label, "hindi": "ERROR", 
            "back_translated": "ERROR", "bleu_score": 0, "quality_flag": "FAIL"
        })

bilingual_df = pd.DataFrame(results)

#  3e. Manual verification 
# TODO: Print 5 random samples for inspection
print("Random samples for manual verification:")
print(bilingual_df.sample(n=5))

#  3f. Save 
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag
bilingual_df.to_csv("bilingual_reviews.csv", index=False)
print("Bilingual reviews saved to bilingual_reviews.csv")

Translating reviews: 100%|██████████| 100/100 [03:33<00:00,  2.13s/it]

Random samples for manual verification:
                                               review     label  \
83  It doesn't leave a lasting impression. It was....   Neutral   
53  Total garbage. Save your money and watch somet...  Negative   
70  The screenplay was frankly pretentious. A frus...  Negative   
45  I want my two hours back. It’s a hard pass fro...  Negative   
44  A frustrating experience. Avoid this at all co...  Negative   

                                                hindi  \
83    यह कोई स्थायी प्रभाव नहीं छोड़ता. यह... ठीक था.   
53         कुल कचरा. अपना पैसा बचाएं और कुछ और देखें।   
70  पटकथा स्पष्ट रूप से दिखावटी थी। एक निराशाजनक अ...   
45  मुझे अपने दो घंटे वापस चाहिए। यह मेरे लिए एक क...   
44          एक निराशाजनक अनुभव. इससे हर कीमत पर बचें.   

                                      back_translated  bleu_score quality_flag  
83  It does not leave any lasting effect. It was.....    0.254509         FAIL  
53  Total garbage. Save your money and watch somet..

In [20]:
print("Total FAILs:", len(bilingual_df[bilingual_df["quality_flag"] == "FAIL"]))
bilingual_df[bilingual_df["quality_flag"] == "FAIL"]

Total FAILs: 21


,review,label,hindi,back_translated,bleu_score,quality_flag
4,"It perfectly balances humor and drama. Wow, si...",Positive,यह हास्य और नाटक को पूरी तरह संतुलित करता है। ...,"It balances comedy and drama perfectly. Wow, j...",0.135460,FAIL
10,Standard fare for this type of movie. Good for...,Positive,इस प्रकार की फ़िल्म के लिए मानक किराया. एक बार...,Standard fare for this type of movie. Nice to ...,0.533432,FAIL
40,Total garbage. Do not bother.,Negative,कुल कचरा. ध्यान न देना।,Total garbage. neglect.,0.191540,FAIL
46,"Neither good nor bad, just exists. It is what ...",Negative,"न अच्छा, न बुरा, बस अस्तित्व में है। जो है सो है।","Neither good, nor bad, just exists. It is what...",0.824237,FAIL
50,A refreshing take on a tired genre. A definiti...,Negative,एक थकी हुई शैली पर एक ताज़ा प्रस्तुति। एक निश्...,A refreshing take on a tired genre. A definite...,0.769161,FAIL
59,An absolute train wreck of a movie. Utterly di...,Negative,किसी फिल्म का बिल्कुल रेल हादसा। एकदम निराशाजनक.,Quite the trainwreck of a movie. Absolutely di...,0.282666,FAIL
60,A cinematic masterpiece. A must-watch for fans...,Negative,एक सिनेमाई उत्कृष्ट कृति. इस शैली के प्रशंसकों...,A cinematic masterpiece. A must-watch for fans...,1.000000,FAIL
61,"Simply put, this is cinema at its finest. Two ...",Negative,सीधे शब्दों में कहें तो यह सिनेमा अपने चरम पर ...,"Simply put, this is cinema at its peak. Two th...",0.614215,FAIL
64,A cinematic masterpiece. Do yourself a favor a...,Negative,एक सिनेमाई उत्कृष्ट कृति. अपने आप पर एक एहसान ...,A cinematic masterpiece. Do yourself a favor a...,0.842363,FAIL
66,A strange experience overall. It’s definitely ...,Negative,कुल मिलाकर एक अजीब अनुभव. यह निश्चित रूप से हर...,Overall a strange experience. It's definitely ...,0.439036,FAIL


As can be seen, there have been a decent number of FAILs in the back-translation. But upon having a closer look into the back translated text for these FAILs, we see that the main crux of the review has very well been kept intact during the back translation. 

These reveals a potential limitation of using BLEU for checking similarities.
- BLEU only counts matching words. A better choice would also be to see if those words can be synonyms. For example impression and effect
- We used short sentences to save tokens in API calls. But this leads to a problem. For shorter reviews, BLEU will be more sensitive to the number of words different. Even a 1-2 word change in a 7-8 word review might lead it to be tagged FAIL
- BLEU also infers doesn't and does not as different, even though grammatically they are the same.

Thus the FAILs which we see here are primarily due to the limitations of BLEU and also due to our strategy of picking shorter reviews.
It is also possible that similar to how the clf was mispredicting for LLM generated reviews, here also since the transalator has a broader vocabulary, some reviews might have been flagged FAIL because of the classifier classifiying it wrongly since it has very limited vocabulary as was trained on only a few hundred reviews.

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [21]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class) 
# TODO: Sample from consolidated_base, mix short and long reviews
audio_samples_df = pd.concat([
    consolidated_df[consolidated_df['label'] == 'Positive'].sample(10, random_state=42),
    consolidated_df[consolidated_df['label'] == 'Negative'].sample(10, random_state=42),
    consolidated_df[consolidated_df['label'] == 'Neutral'].sample(10, random_state=42)
]).reset_index(drop=True)


#  4b. TTS generation 
os.makedirs("audio_samples", exist_ok=True)

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile
wav_files = []
for idx, row in tqdm(audio_samples_df.iterrows(), total=len(audio_samples_df), desc="Generating audio samples"):
    review = row["review"]
    label = row["label"]

    try:
        tts = gTTS(text=review, lang='en', tld='com')
        mp3_path = f"audio_samples/review_{idx}.mp3"
        wav_path = f"audio_samples/review_{idx}.wav"
        tts.save(mp3_path)
        
        # Load mp3 with librosa and save as wav
        y, sr = librosa.load(mp3_path, sr=None)
        sf.write(wav_path, y, sr)
        wav_files.append((wav_path, label))
    except Exception as e:
        print(f"Error generating audio for index {idx}: {e}")

print(f"Generated {len(wav_files)} audio samples.")

Generating audio samples: 100%|██████████| 30/30 [00:46<00:00,  1.54s/it]

Generated 30 audio samples.


In [22]:
#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv
audio_features = []
for wav_path, label in tqdm(wav_files, desc="Extracting audio features"):
    try:
        y, sr = librosa.load(wav_path, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)
        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zero_crossing_rate = np.mean(librosa.feature.zero_crossing_rate(y))
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)  # Take mean across time
        
        audio_features.append({
            "wav_path": wav_path,
            "label": label,
            "duration": duration,
            "spectral_centroid": spectral_centroid,
            "zero_crossing_rate": zero_crossing_rate,
            **{f"mfcc_{i+1}": mfccs_mean[i] for i in range(13)}
        })
    except Exception as e:
        print(f"Error processing {wav_path}: {e}")

audio_features_df = pd.DataFrame(audio_features)
audio_features_df.to_csv("audio_features.csv", index=False)

Extracting audio features: 100%|██████████| 30/30 [00:00<00:00, 32.04it/s]


In [23]:
audio_features_df

,wav_path,label,duration,spectral_centroid,zero_crossing_rate,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13
0,audio_samples/review_0.wav,Positive,7.920,2404.892471,0.114687,-331.947876,70.522888,7.851284,13.842457,-3.293601,-2.230688,0.340447,-4.210880,-9.209472,-2.447442,-8.814178,-1.503038,-10.734947
1,audio_samples/review_1.wav,Positive,5.208,1883.148936,0.073254,-333.017548,89.802536,-0.023211,17.362368,6.694998,-9.112065,-3.507668,-10.228398,-10.194213,-4.937937,-12.237619,-0.663420,-13.793406
2,audio_samples/review_2.wav,Positive,5.592,2231.755094,0.099626,-333.339478,73.587509,10.697739,18.139309,10.549703,-5.656809,-6.519043,-6.192922,-10.316793,-7.170886,-13.779021,-1.145927,-12.212165
3,audio_samples/review_3.wav,Positive,5.712,2090.381801,0.088124,-336.452728,77.644005,11.386763,17.599245,3.937860,-3.703098,-2.454534,-9.546782,-9.312916,-6.249987,-10.781871,-1.153988,-12.530995
4,audio_samples/review_4.wav,Positive,9.576,2291.529921,0.109610,-324.502777,82.537743,10.794256,17.286821,-1.036109,-3.819144,-2.830416,-9.085018,-9.915356,-5.322644,-13.594677,-1.572221,-12.745092
5,audio_samples/review_5.wav,Positive,10.152,2395.344712,0.114470,-336.840393,73.792633,13.528509,20.128775,-0.506445,-4.761104,-2.982109,-6.440333,-9.964475,-4.682991,-12.904553,-0.820822,-11.500786
6,audio_samples/review_6.wav,Positive,4.776,2160.289791,0.104050,-350.323242,77.948776,15.828398,15.898828,6.677061,-1.732837,-1.929664,-7.863251,-11.119285,-6.061510,-12.828393,-1.598070,-13.772069
7,audio_samples/review_7.wav,Positive,8.256,2187.558896,0.096439,-336.595551,83.855255,6.655260,11.218194,-7.066481,-4.552009,1.962076,-6.061650,-10.702826,-3.255731,-8.816374,-1.630503,-12.517024
8,audio_samples/review_8.wav,Positive,8.376,2372.911009,0.121340,-339.455322,75.359863,15.771159,18.458157,3.044678,-3.523920,-3.379369,-6.123755,-10.820736,-7.775308,-12.329425,-0.258176,-12.228285
9,audio_samples/review_9.wav,Positive,4.680,2315.334473,0.097854,-351.791687,67.636719,16.511187,15.720872,-1.709204,-3.178349,-1.076000,-2.382561,-9.314697,-4.417547,-8.994760,-1.974074,-11.765512


In [27]:
#  4d. Whisper round-trip validation 
import whisper

_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv

def wer(word1, word2):
    words1 = word1.split()
    words2 = word2.split()
    d = np.zeros((len(words1)+1, len(words2)+1), dtype=np.uint8)
    for i in range(len(words1)+1):
        d[i][0] = i
    for j in range(len(words2)+1):
        d[0][j] = j
    for i in range(1, len(words1)+1):
        for j in range(1, len(words2)+1):
            if words1[i-1] == words2[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = min(d[i-1][j] + 1,      # Deletion
                              d[i][j-1] + 1,      # Insertion
                              d[i-1][j-1] + 1)   # Substitution
    return d[len(words1)][len(words2)] / max(len(words1), 1)

validation_results = []

base_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
audio_folder = os.path.join(base_dir, "audio_samples")

for idx, row in tqdm(audio_features_df.iterrows(), total=len(audio_features_df), desc="Validating audio with Whisper"):
    wav_path = os.path.join(audio_folder, f"review_{idx}.wav")
    label = row["label"]
    try:
        result = _whisper_model.transcribe(wav_path)
        transcription = result["text"].strip()
        
        original_idx = int(os.path.basename(wav_path).split("_")[1].split(".")[0])
        original_review = audio_samples_df.iloc[original_idx]["review"]

        word1 = re.sub(r"[^\w\s]", "", original_review).lower()
        word2 = re.sub(r"[^\w\s]", "", transcription).lower()

        error_rate = wer(word1, word2)
        quality_flag = "PASS" if error_rate <= 0.25 else "FAIL"
        
        validation_results.append({
            "wav_path": wav_path,
            "label": label,
            "original_review": original_review,
            "transcription": transcription,
            "wer": error_rate,
            "quality_flag": quality_flag
        })
    except Exception as e:
        print(f"Error validating {wav_path}: {e}")
        validation_results.append({
            "wav_path": wav_path,
            "label": label,
            "original_review": "ERROR",
            "transcription": "ERROR",
            "wer": 1.0,
            "quality_flag": "FAIL"
        })

validation_results_df = pd.DataFrame(validation_results)
#Count of PASS and FAIL
print("Validation results:\n", validation_results_df["quality_flag"].value_counts())
validation_results_df.to_csv("audio_validation.csv", index=False)

Validating audio with Whisper: 100%|██████████| 30/30 [00:23<00:00,  1.27it/s]

Validation results:
 quality_flag
PASS    29
FAIL     1
Name: count, dtype: int64


Very few reviews FAIl because issues like 10/10 returned as 10 out of 10 which causes a high penalty in WER for small reviews. Otherwise the results are decently good.

## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [25]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv


#  5b. Black-Box Evaluation 
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)

# Print comparison
# print(f"Baseline accuracy:  {baseline_acc:.2%}")
# print(f"Augmented accuracy: {augmented_acc:.2%}")
# print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")

Initializing Black-Box Embedder...


ValueError: The provided filename text_embedder.pt does not exist